# 00 · Contrato y auditoría — Economic Intelligence OS
**Objetivo:** demostrar qué dato es histórico observado, qué dato es referencia actual y qué puede entrar a ML sin fuga de información.

Regla central: `stock_total_departamentos_actual_ref` es el universo completo **actual** del proyecto; sirve para cobertura y planeamiento, pero no se retroproyecta como stock histórico de lanzamiento.

In [ ]:
from pathlib import Path
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'replica_cygnus').exists())
sys.path.insert(0, str(ROOT/'src'))
from replica_cygnus.economic_intelligence import install_feature_mart, load_monthly_panel
install_feature_mart()
panel = load_monthly_panel()
panel.shape, panel['proyecto'].nunique(), panel['periodo_mes'].min(), panel['periodo_mes'].max()

## Gate de calidad
Antes de modelar: continuidad temporal, identidad de stock, cobertura del ledger y missingness. Los meses sin evidencia no se convierten en ceros.

In [ ]:
audit = (panel.groupby(['codigo_proyecto','proyecto'], as_index=False)
         .agg(primer_mes=('periodo_mes','min'), ultimo_mes=('periodo_mes','max'), meses=('periodo_mes','nunique'),
              stock_total_ref=('stock_total_departamentos_actual_ref','max'),
              stock_ofertado_obs=('stock_ofertado_observado_acum','max'),
              cobertura_ledger=('cobertura_oferta_ledger_vs_universo_actual','max')))
audit['gap_universo_vs_ledger'] = audit['stock_total_ref'] - audit['stock_ofertado_obs']
audit.sort_values('cobertura_ledger').head(20)

In [ ]:
check = panel.copy()
check['saldo_identidad'] = check['stock_inicio_observado'] + check['altas_mes'] - check['separaciones_brutas_mes'] + check['caidas_mes']
check['gap_identidad_stock'] = check['saldo_final_observado'] - check['saldo_identidad']
check.loc[check['gap_identidad_stock'].fillna(0).abs() > 1e-9, ['proyecto','periodo_mes','gap_identidad_stock']].head(30)

In [ ]:
missing = panel.isna().mean().sort_values(ascending=False).rename('pct_missing').to_frame()
missing.head(25)

## Decisión de gobierno
**Apto para entrenamiento histórico:** flujos observados, lags, stock observado, edad comercial, estacionalidad y agregados de mercado.

**Sólo referencia/escenario:** precios actuales, descuentos actuales y universo actual completo. No usar como si fueran precios históricos salvo que se incorpore un snapshot de precios por mes.